#### Event Data Ingestion — Ticketmaster Discovery API

The event layer is fetched from the **Ticketmaster Discovery API** — a live event discovery service providing venue, classification, and scheduling data for public events across Berlin. Each of the three districts is queried independently using its central coordinates with a 10km radius to capture all relevant venues within the operational zone.

**Time Window:**
Unlike the trip logs and weather data which cover a historical 3-week window, the events layer is **forward-looking** — fetching the upcoming 21 days from the current execution date. This is a deliberate architectural decision: Ticketmaster's Discovery API does not serve historical event data, only scheduled upcoming events. This shifts the events layer from a retrospective analysis tool to a **predictive rebalancing signal** — enabling fleet managers to pre-position drivers ahead of known demand surges rather than reacting after the fact.

**Radius & Deduplication:**
Each district is queried with a 10km radius, which means events near district boundaries may appear in multiple queries. A `drop_duplicates` on `event_id` is applied before export to ensure each event is represented exactly once in the dataset.

**Metrics retained per event:**
- `event_id` — unique Ticketmaster identifier
- `event_name` — name of the event
- `event_type` — segment classification (Music, Sports, Arts, etc.)
- `venue` — venue name
- `event_date` — scheduled start datetime (UTC)
- `timestamp_hour` — event date floored to the nearest hour for BigQuery join alignment
- `district` — the Berlin district the event is assigned to
- `latitude` / `longitude` — district centroid coordinates

**Join Strategy:**
The events table joins to the staging layer on `timestamp_hour` and `district` using a `LEFT JOIN`, preserving all trip records regardless of whether an event was occurring. A boolean flag `is_event_hour` is derived at the analytical layer to compare fleet behavior during event periods versus non-event periods.

In [2]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [41]:
from dotenv import load_dotenv
import os

os.environ.pop('TICKETMASTER_API_KEY', None)
load_dotenv(override=True)
API_KEY = os.getenv('TICKETMASTER_API_KEY')
print(f"Key length: {len(API_KEY)}")  # should print 32

Key length: 32


In [45]:
import requests
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os

load_dotenv(override=True)
API_KEY = os.getenv('TICKETMASTER_API_KEY')

# 1. Configuration
BERLIN_DISTRICTS = {
    'Mitte': {'lat': 52.5200, 'lon': 13.4050},
    'Prenzlauer Berg': {'lat': 52.5382, 'lon': 13.4237},
    'Friedrichshain-Kreuzberg': {'lat': 52.5002, 'lon': 13.4494}
}

# 2. Upcoming 21-day window
start_str = datetime.now().strftime('%Y-%m-%dT%H:%M:%SZ')
end_str = (datetime.now() + timedelta(days=21)).strftime('%Y-%m-%dT%H:%M:%SZ')

# 3. Fetch events per district
all_events = []

for district, coords in BERLIN_DISTRICTS.items():
    print(f"Fetching events for {district}...")

    url = "https://app.ticketmaster.com/discovery/v2/events.json"
    params = {
        'apikey': API_KEY,
        'latlong': f"{coords['lat']},{coords['lon']}",
        'radius': '10',
        'unit': 'km',
        'startDateTime': start_str,
        'endDateTime': end_str,
        'size': 200,
        'countryCode': 'DE'
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:
        data = response.json()
        events = data.get('_embedded', {}).get('events', [])

        for event in events:
            all_events.append({
                'event_id': event.get('id'),
                'event_name': event.get('name'),
                'district': district,
                'event_date': event.get('dates', {}).get('start', {}).get('dateTime'),
                'event_type': event.get('classifications', [{}])[0].get('segment', {}).get('name', 'Unknown'),
                'venue': event.get('_embedded', {}).get('venues', [{}])[0].get('name', 'Unknown'),
                'latitude': coords['lat'],
                'longitude': coords['lon']
            })

        print(f"  Found {len(events)} events in {district}")
    else:
        print(f"  Failed for {district}: {response.status_code} — {response.text}")

# 4. Build DataFrame
df_events = pd.DataFrame(all_events)

if not df_events.empty:
    # 5. Standardize timestamp
    df_events['event_date'] = pd.to_datetime(df_events['event_date'], utc=True)
    df_events['timestamp_hour'] = df_events['event_date'].dt.floor('H')

    # 6. Remove duplicates — same event can appear in multiple district radius searches
    df_events = df_events.drop_duplicates(subset='event_id')

    # 7. Export
    output_filename = 'berlin_events_raw.csv'
    df_events.to_csv(output_filename, index=False)
    print(f"\nSuccess! Saved {len(df_events)} events to '{output_filename}'")
    print("\nFirst 5 rows preview:")
    print(df_events.head())
else:
    print("\nNo events found — check your API key and date window")

Fetching events for Mitte...
  Found 72 events in Mitte
Fetching events for Prenzlauer Berg...
  Found 72 events in Prenzlauer Berg
Fetching events for Friedrichshain-Kreuzberg...
  Found 72 events in Friedrichshain-Kreuzberg

Success! Saved 72 events to 'berlin_events_raw.csv'

First 5 rows preview:
             event_id                                      event_name  \
0   Z698xZC2Z1kJ6oxjw                 Guns N’ Roses - World Tour 2026   
1   Z698xZC2Z1kfFb8jz  Guns N’ Roses - World Tour 2026 | VIP Packages   
2  Z698xZC2Z16v0ojMag  Guns N’ Roses - World Tour 2026 | VIP Packages   
3  Z698xZC2Z16vPZoO16      Guns N’ Roses - World Tour 2026 | Box seat   
4   Z698xZC2Z1kCfCdGV                 Guns N’ Roses - World Tour 2026   

  district                event_date event_type       venue  latitude  \
0    Mitte 2026-06-23 16:30:00+00:00      Music  Uber Arena     52.52   
1    Mitte 2026-06-23 16:30:00+00:00      Music  Uber Arena     52.52   
2    Mitte 2026-06-25 16:30:00+00:00    

/var/folders/2g/qs15k_xx6_l4ycvkj3d13ct80000gn/T/ipykernel_44939/1184554011.py:67: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_events['timestamp_hour'] = df_events['event_date'].dt.floor('H')


In [46]:
df_events

,event_id,event_name,district,event_date,event_type,venue,latitude,longitude,timestamp_hour
0,Z698xZC2Z1kJ6oxjw,Guns N’ Roses - World Tour 2026,Mitte,2026-06-23 16:30:00+00:00,Music,Uber Arena,52.52,13.405,2026-06-23 16:00:00+00:00
1,Z698xZC2Z1kfFb8jz,Guns N’ Roses - World Tour 2026 | VIP Packages,Mitte,2026-06-23 16:30:00+00:00,Music,Uber Arena,52.52,13.405,2026-06-23 16:00:00+00:00
2,Z698xZC2Z16v0ojMag,Guns N’ Roses - World Tour 2026 | VIP Packages,Mitte,2026-06-25 16:30:00+00:00,Music,Uber Arena,52.52,13.405,2026-06-25 16:00:00+00:00
3,Z698xZC2Z16vPZoO16,Guns N’ Roses - World Tour 2026 | Box seat,Mitte,2026-06-25 16:30:00+00:00,Music,Uber Arena,52.52,13.405,2026-06-25 16:00:00+00:00
4,Z698xZC2Z1kCfCdGV,Guns N’ Roses - World Tour 2026,Mitte,2026-06-25 16:30:00+00:00,Music,Uber Arena,52.52,13.405,2026-06-25 16:00:00+00:00
...,...,...,...,...,...,...,...,...,...
67,LvZ18QKZI-ODi2YZycYN_,Bottoms Up Brunch!,Mitte,2026-06-27 10:00:00+00:00,Arts & Theatre,Vagabund Brauerei Kesselhaus,52.52,13.405,2026-06-27 10:00:00+00:00
68,Z698xZC2Z1kCK7kbV,Casey Frey and Special Guest – Nick Colletti,Mitte,2026-06-18 18:00:00+00:00,Arts & Theatre,Unknown,52.52,13.405,2026-06-18 18:00:00+00:00
69,LvZ18QbNSsQ6VUZZv7Vkk,Lachkater Open Air Stand Up Comedy Show Luftsc...,Mitte,2026-06-19 17:30:00+00:00,Arts & Theatre,Luftschloss Tempelhofer Feld,52.52,13.405,2026-06-19 17:00:00+00:00
70,Z698xZC2Z16vA68e4J,femtanyl - Man Bites Dog Tour,Mitte,2026-06-15 18:00:00+00:00,Music,Unknown,52.52,13.405,2026-06-15 18:00:00+00:00
